In [27]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("SPTRANS_TOKEN")

BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

print("Token encontrado:", TOKEN is not None)

Token encontrado: True


In [28]:
sessao = requests.Session()

resposta = sessao.post(
    f"{BASE_URL}/Login/Autenticar",
    params={"token": TOKEN},
    timeout=15
)

print("Código HTTP:", resposta.status_code)
print("Resposta:", resposta.text)

Código HTTP: 200
Resposta: true


In [29]:
resposta = sessao.get(f"{BASE_URL}/Linha/Buscar", params={"termosBusca": "8000"}, timeout=15) # Na variável resposta, vou pegar(Get) dentro da Base URl, o end-point 
linhas = resposta.json()  # transforma o texto da resposta em lista do Python

tabela_linhas = pd.DataFrame(linhas) # Peguei a variavel LINHAS e transformei esse resultado em um DataFrame para se tornar uma tabela.
tabela_linhas

,cl,lc,lt,sl,tl,tp,ts
0,2506,False,8000,1,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
1,35274,False,8000,2,1,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
2,1273,False,8000,1,10,PÇA. RAMOS DE AZEVEDO,TERM. LAPA
3,34041,False,8000,2,10,PÇA. RAMOS DE AZEVEDO,TERM. LAPA


In [30]:
codigo_linha = 35274

resposta = sessao.get(f"{BASE_URL}/Parada/BuscarParadasPorLinha", # Chamando(Get) End-point da Base_URL para retornar informações de Paradas das Linhas.
                      params={"codigoLinha": codigo_linha}, timeout=15) # Passei como parametro que a resposta deve receber informações guardadas no código da variável codigo_linha.
paradas = pd.DataFrame(resposta.json()) # Estou transformando a resposta que veio em HTTPS em uma lista Json (Dicionário) e trasformando o Json em uma "DF" de Tabela, guardando em uma variável.

paradas = paradas.rename(columns={   # Utilizando o ".rename", eu tratei nomes das colunas para ficar mais prático de análisar olhando os nomes.
    "cp": "codigo_parada",
    "np": "nome_parada",
    "ed": "endereco",
    "py": "latitude",
    "px": "longitude"
})

print("Quantidade de paradas:", len(paradas)) #Linha Básica para verificar o tamanho da tabela paradas utilzando "Len"
paradas.head(15)  # Por tendência o ".head" pega as 5 primeiras linhas da tabela, mas podemos passar no parametro quantas linhas podemos ver.

Quantidade de paradas: 13


,codigo_parada,nome_parada,endereco,latitude,longitude
0,614432,BABY BARIONY C/B,R PEDRO MACHADO,-23.527606,-46.671514
1,7014416,ANGELICA C/B,R DR. ALBUQUERQUE LINS,-23.534377,-46.654322
2,60014604,SESC POMPEIA C/B,VDTO POMPEIA/ R JOAQUIM FERREIRA,-23.525262,-46.682254
3,60016783,PARADA PALMEIRAS C/B,R PADRE ANTONIO TOMAS/ AV POMPEIA,-23.525653,-46.679493
4,60016785,ANTARTICA C/B,R ENGENHEIRO STEVENSON/ PC SOUSA ARANHA,-23.526220,-46.674187
5,60016789,PQ. DA ÁGUA BRANCA C/B,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,-23.529636,-46.667777
6,480014608,TIBERIO C/B,R TIBERIO/ R MENFIS,-23.522875,-46.688219
7,480014609,DUILIO C/B,R DUILIO/ R CLAUDIO,-23.522627,-46.692121
8,480014610,TENDAL DA LAPA C/B,R ESPARTACO/ R MORAIS E SILVA,-23.522339,-46.696246
9,700006085,ROSA E SILVA C/B,R OLIMPIA DE ALMEIDA PRADO/ R LOPES DE OLIVEIRA,-23.532629,-46.657465


In [31]:
from math import radians, sin, cos, asin, sqrt # Importei da biblioteca Math, algumas operações matemáticas que vou precisar.

def distancia_m(lat1, lon1, lat2, lon2):
    """Distância em metros entre dois pontos (fórmula de Haversine)."""
    Raio = 6371000  # O numero atribuído a variável representa o raio da Terra em metros
    dlat = radians(lat2 - lat1) # Guarda o resultado passado para Radianos da conta de (Lat2 - Lat1) que está cotada em Graus.
    dlon = radians(lon2 - lon1) # Guarda o resultado passado para Radianos da conta de (Lon2 - Lon1) que está cotada em Graus.
    
    
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * Raio * asin(sqrt(a))

In [32]:
p1 = paradas.iloc[0]   # Aqui eu estou localizando a Primeira parada que existe na tabela, Guardei essa parada em uma variável.
p2 = paradas.iloc[1]   # segunda parada

d = distancia_m(p1["latitude"], p1["longitude"], p2["latitude"], p2["longitude"]) # Chamando a função criada e instanciando essa função com os parametros das informações sobre as Paradas como: Longitude e Latitude.
print(f"Distância entre '{p1['nome_parada']}' e '{p2['nome_parada']}': {d:.0f} metros") # Pegando as informações das Paradas e comparando a distancia entre elas chamando a variável "D" que possui o resultado calculado.

Distância entre 'BABY BARIONY C/B' e 'ANGELICA C/B': 1908 metros


In [33]:

distancias = [] # Criando uma variável de lista para guardar as distancias
for i in range(1, len(paradas)): # Loop para pegar cada Parada da tabela Paradas
    anterior = paradas.iloc[i - 1] # Guardando o valor anterior do valor atual (i) do loop na variável "anterior".
    atual = paradas.iloc[i] # Guardando o valor atual do Loop (i) na variável "atual"
    distancias.append( # O append aqui serve para guardar na lista "distancia".
        distancia_m(anterior["latitude"], anterior["longitude"], # Dentro do append, chamei a função para calcular os parametros recebidos da latitude e longitude das variaveis "anterior" e "atual".
                    atual["latitude"], atual["longitude"])
    )

serie = pd.Series(distancias) # Guardei em uma variável e tranformei a lista "distancia" em uma Série com indices e valores para ánalisa melhor.
print(serie.describe()) # O describe serve para fazer uma análise Estatística sobre a Série "Distancias"
print("Saltos maiores que 1000 m:", (serie > 1000).sum(), "de", len(serie)) # Retornei utilizando um filtro, a contagem de distancias de são > 1000 e comparei com o tamanho da lista utilizando "Len".


count      12.000000
mean     1467.166558
std      1180.944655
min       284.829257
25%       513.911070
50%      1190.791606
75%      1984.562957
max      4116.000587
dtype: float64
Saltos maiores que 1000 m: 7 de 12


In [ ]:
def paradas_proximas(tabela_paradas, lat, lon, raio_max): # Essa função serve para encontrar as paradas mais próximas do ponto de origem.
    """Devolve as paradas a até raio_max metros do ponto, da mais perto para a mais longe."""
    
    tabela = tabela_paradas.copy() # Copia o DataFrame original e salva em outra variável "teste", para não alterar o DataFrame Original.
    tabela["distancia_m"] = tabela.apply( # Estou criando uma coluna chamada "distancia_m" na nova tabela e estou aplicando uma Função com o apply.
        lambda parada: distancia_m(lat, lon, parada["latitude"], parada["longitude"]), axis=1 # Lamba é uma mini função dentro do apply que vai chamar a função distancia_m e passar os parametros das cordenadas do usuário e das paradas.
    ) # o "Axis = 1" significa que vai trabalhar o lambda pegando linha por linha, se fosse "0" seria para pegar de coluna em coluna
    
    return tabela[tabela["distancia_m"] <= raio_max].sort_values("distancia_m") # Aplicando um filtro para retornar uma tabela com apenas as distâncias que atendem o parametro de "Raio_max", e ordenando esses valores de forma ascentente.

In [38]:
lat_teste = paradas.iloc[0]["latitude"] # Recebendo a latitude da parada no indice 0 na tabela "Paradas"
lon_teste = paradas.iloc[0]["longitude"] # Recebendo a longitude da parada no indice 0 na tabela "Paradas"

perto = paradas_proximas(paradas, lat_teste, lon_teste, 500) # Aqui eu estou guardando em uma variável as intâncias que a função vai receber, recebendo o DataFrame, as variáveis acima e o Raio_max.
print("Paradas encontradas:", len(perto)) # Usando Len para conferir a quantidade de distancias que possuem dentro da variável "perto"
display (perto[["nome_parada", "endereco", "distancia_m"]]) # Retornando as as colunas com as distancias que atendem aos requisitos.

Paradas encontradas: 3


,nome_parada,endereco,distancia_m
0,BABY BARIONY C/B,R PEDRO MACHADO,0.000000
4,ANTARTICA C/B,R ENGENHEIRO STEVENSON/ PC SOUSA ARANHA,313.076812
5,PQ. DA ÁGUA BRANCA C/B,R DONA GERMAINE BUCHARD/ R MINISTRO GODOI,442.836379
